In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from flask import Flask, request, jsonify

# --- Servicios internos ---
from services.image_io import decode_image_from_request
from services.onnx_detector import ONNXFaceDetector
from services.uniface_pipeline import recognize_primary_face
# Add these imports:
from services.external_client import ExternalClient
from services.matching import find_best_match

# ----------------------------------------------------
# Inicialización Flask
# ----------------------------------------------------
app = Flask(__name__)

# ----------------------------------------------------
# Inicialización modelos (UNA SOLA VEZ)
# ----------------------------------------------------
onnx_detector = ONNXFaceDetector(
    model_path="models/face_detector.onnx"
)
# Initialize the client:
external_client = ExternalClient()

In [3]:
# ----------------------------------------------------
# Healthcheck
# ----------------------------------------------------
@app.route("/health", methods=["GET"])
def health():
    return jsonify(status="ok"), 200


# ----------------------------------------------------
# /detect
# SOLO detección rápida (ONNX)
# ----------------------------------------------------
@app.route("/detect", methods=["POST"])
def detect():
    """
    Detección rápida con modelo ONNX.
    NO reconocimiento.
    """
    image = decode_image_from_request(request)
    if image is None:
        return jsonify(error="No image received"), 400

    result = onnx_detector.detect(image, thresh=0.5)

    if result is None:
        return jsonify(
            detected=False,
            reason="no_face_detected"
        ), 200

    return jsonify(
        detected=True,
        score=result["score"],
        bbox=result["bbox"]  # [x1, y1, x2, y2]
    ), 200


# ----------------------------------------------------
# /recognize
# Detecta TODAS → selecciona UNA → ArcFace
# ----------------------------------------------------
@app.route("/recognize", methods=["POST"])
def recognize():
    """
    Pipeline correcto de reconocimiento facial:
    - ONNX (early exit)
    - RetinaFace (todas las caras)
    - Selección de cara principal
    - ArcFace con landmarks
    """
    image = decode_image_from_request(request)
    if image is None:
        return jsonify(error="No image received"), 400

    # --- Pipeline UniFace correcto ---
    result, error = recognize_primary_face(image)

    if error:
        return jsonify(
            recognized=False,
            reason=error
        ), 200

    return jsonify(
        recognized=True,
        embedding=result["embedding"],       # 512 floats
        primary_face=result["primary_face"]  # bbox + scores
    ), 200


# ----------------------------------------------------
# Main
# ----------------------------------------------------
if __name__ == "__main__":
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False
    )


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.17.82.192:5000
Press CTRL+C to quit
127.0.0.1 - - [09/Jan/2026 11:04:44] "POST /detect HTTP/1.1" 200 -


DEBUG: decode_image_from_request entered
DEBUG: req.files keys: ['image']
DEBUG: Found file_storage, reading...
DEBUG: Read 34896 bytes
DEBUG: Decoded image shape: (576, 1024, 3)
DEBUG: decode_image_from_request entered
DEBUG: req.files keys: []


127.0.0.1 - - [09/Jan/2026 11:06:44] "POST /recognize HTTP/1.1" 400 -
127.0.0.1 - - [09/Jan/2026 11:10:00] "POST /detect HTTP/1.1" 200 -


DEBUG: decode_image_from_request entered
DEBUG: req.files keys: ['image']
DEBUG: Found file_storage, reading...
DEBUG: Read 34896 bytes
DEBUG: Decoded image shape: (576, 1024, 3)
DEBUG: decode_image_from_request entered
DEBUG: req.files keys: []


127.0.0.1 - - [09/Jan/2026 11:10:48] "POST /detect HTTP/1.1" 200 -


DEBUG: decode_image_from_request entered
DEBUG: req.files keys: ['image']
DEBUG: Found file_storage, reading...
DEBUG: Read 34896 bytes
DEBUG: Decoded image shape: (576, 1024, 3)
DEBUG: decode_image_from_request entered
DEBUG: req.files keys: []


127.0.0.1 - - [09/Jan/2026 11:12:00] "POST /recognize HTTP/1.1" 400 -
127.0.0.1 - - [09/Jan/2026 11:12:48] "POST /recognize HTTP/1.1" 400 -
